# Feature Derivation Notebook

This notebook derives new features from the imputed dataset to enhance model performance.

## Overview
- **Input**: `missing_values_imputed.csv`
- **Output**: `features_derived.csv`
- **Key Operations**:
  1. Derive time-based features (weeks between events)
  2. Create categorical features from continuous variables
  3. Generate student enrollment indicators
  4. Handle structural missingness with flags
  5. Clean redundant columns

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import warnings
from IPython.display import display
from collections import defaultdict

warnings.filterwarnings('ignore')
pd.options.display.max_columns = None

## 1. Setup and Configuration

### 1.1 Import Libraries

In [ ]:
# Define paths
current_user = os.getlogin()
DATA_DIR = Path(f"/home/{current_user}/local-share")
OUT_DIR = DATA_DIR / "processed"

# Load imputed data
data = pd.read_csv(OUT_DIR / "missing_values_imputed.csv", sep=None, engine="python", encoding="utf-8-sig")
print(f"Data shape: {data.shape}")
data.head()

## 2. Feature Derivation Plan

This section derives 7 new features from the dataset to capture temporal relationships, distance changes, and categorical groupings.

In [ ]:
# =========================================================================
# FEATURE DERIVATION PLAN
# =========================================================================
# 
# 1. change_in_distance
#    Δ distance (km) between previous and current address to campus (3584_CS)
#    Formula: distance_current − distance_previous
#
# 2. age_category
#    Age groups at degree start:
#    • <18 → 'early_starters'
#    • 18–19 → 'common_student_age'
#    • 20–22 → 'late_starters'
#    • >22 → 'career_changers'
#
# 3. time_first_orient_to_start_weeks
#    Weeks between first orientation event and degree start
#
# 4. time_last_orient_to_start_weeks
#    Weeks between last orientation event and degree start
#
# 5. gap_skc_to_start_weeks
#    Weeks between SKC meeting and degree start (can be ± due to timing)
#
# 6. time_enroll_to_start_weeks
#    Weeks between enrollment date and degree start
#
# 7. gap_prev_exam_to_start_weeks
#    Weeks between previous education final exam and degree start
# =========================================================================

### 2.1 Helper Functions

In [ ]:
def _get_dt(df: pd.DataFrame, col: str) -> pd.Series:
    """Safely convert column to datetime, return NaT if missing."""
    s = df.get(col)
    return pd.Series(pd.NaT, index=df.index) if s is None else pd.to_datetime(s, errors='coerce')

def _get_num(df: pd.DataFrame, col: str) -> pd.Series:
    """Safely convert column to numeric, return NaN if missing."""
    s = df.get(col)
    return pd.Series(np.nan, index=df.index) if s is None else pd.to_numeric(s, errors='coerce')

def _weeks_between(later: pd.Series, earlier: pd.Series) -> pd.Series:
    """Calculate weeks between two dates (later - earlier). Preserves sign."""
    dt = pd.to_datetime(later, errors='coerce') - pd.to_datetime(earlier, errors='coerce')
    return (dt / pd.Timedelta(weeks=1)).astype(float)

def derive_features(df, age_bins=None, age_labels=None, rounding=2) -> pd.DataFrame:
    """
    Derive 7 new features from the dataset.
    Robust to missing columns (returns NaN/NaT where unavailable).
    """
    df_out = df.copy()

    # Extract numeric columns
    dist_curr = _get_num(df_out, 'sdo1_postal_distance_distance_to_3584_CS')
    dist_prev = _get_num(df_out, 'sdo1_prev_distance_distance_to_3584_CS')
    age_start = _get_num(df_out, 'sdo1_characteristics_age_start_study')

    # Extract date columns
    deg_start = _get_dt(df_out, 'sdo5_degree_degree_start_date')
    first_orient = _get_dt(df_out, 'sdo2_orientation_First_Event_Date')
    last_orient = _get_dt(df_out, 'sdo2_orientation_Last_Event_Date')
    skc_date = _get_dt(df_out, 'sdo2_skc_SKC_DATUM')
    enroll_date = _get_dt(df_out, 'sdo5_degree_enrollment_date')
    prev_exam_date = _get_dt(df_out, 'sdo1_previous_Final_Exam_Date')

    # 1. Distance change
    df_out['change_in_distance'] = (dist_curr - dist_prev).round(rounding)

    # 2. Age category
    if age_bins is None:
        age_bins = [-np.inf, 18, 20, 23, np.inf]
    if age_labels is None:
        age_labels = ['early_starters', 'common_student_age', 'late_starters', 'career_changers']
    
    df_out['age_category'] = pd.cut(age_start, bins=age_bins, labels=age_labels, 
                                     right=False, include_lowest=True)

    # 3-7. Time deltas in weeks
    df_out['time_first_orient_to_start_weeks'] = _weeks_between(deg_start, first_orient).round(rounding)
    df_out['time_last_orient_to_start_weeks'] = _weeks_between(deg_start, last_orient).round(rounding)
    df_out['gap_skc_to_start_weeks'] = _weeks_between(deg_start, skc_date).round(rounding)
    df_out['time_enroll_to_start_weeks'] = _weeks_between(deg_start, enroll_date).round(rounding)
    df_out['gap_prev_exam_to_start_weeks'] = _weeks_between(deg_start, prev_exam_date).round(rounding)

    # Preserve has_orientation
    if 'has_orientation' in df_out.columns:
        df_out['has_orientation'] = df_out['has_orientation'].astype('Int64')

    return df_out

### 2.2 Apply Feature Derivation

In [ ]:
# Derive features
data = derive_features(data, rounding=2)

# Preview derived features
display(data.filter(regex="change_in_distance|age_category|_weeks$|^has_").head(10))

# Summary statistics for time deltas
print("\n📊 Summary of time deltas (weeks):")
print(data.filter(regex="_weeks$").describe().T)

### 2.3 Investigate Negative SKC Gap Values

In [ ]:
# Inspect negative SKC gaps (SKC occurred after degree start)
mask = data['gap_skc_to_start_weeks'] < 0

print(f"Negative SKC gaps: {mask.sum():,} ({mask.sum()/len(data)*100:.1f}%)")
print("\nExamples (sorted by gap):")
display(
    data.loc[mask, ['sdo5_degree_degree_start_date', 'sdo2_skc_SKC_DATUM', 'gap_skc_to_start_weeks']]
    .sort_values('gap_skc_to_start_weeks').head(20)
)

print("\n📊 gap_skc_to_start_weeks distribution:")
print(data['gap_skc_to_start_weeks'].describe())

## 4. Student Enrollment Indicators

### 4.1 Multiple Degrees Enrollment Indicator

In [ ]:
# Count unique degrees per student-year
degree_counts = (
    data.groupby(['sdo1_characteristics_student_number', 'sdo5_degree_COLLEGEJAAR'])
    ['sdo5_degree_degree']
    .nunique()
    .reset_index(name='degree_count_year')
)

# Create binary indicator (1 if >1 degree in same year)
degree_counts['Multiple_Degrees_Enrollment'] = (degree_counts['degree_count_year'] > 1).astype(int)

# Merge back to main dataframe
data = data.merge(
    degree_counts[['sdo1_characteristics_student_number', 'sdo5_degree_COLLEGEJAAR', 
                   'Multiple_Degrees_Enrollment']],
    on=['sdo1_characteristics_student_number', 'sdo5_degree_COLLEGEJAAR'],
    how='left'
)

print("📊 Multiple Degrees Enrollment distribution:")
print(data['Multiple_Degrees_Enrollment'].value_counts(dropna=False))

## 5. Data Quality Checks and Cleanup

### 5.1 Check for Duplicate Columns

In [ ]:
# =========================================================================
# REDUNDANCY CHECKS
# =========================================================================
# Check previous-school postcode columns
if 'sdo1_prev_distance_postcode' in data and 'sdo1_previous_Previous_School_Postal_Code' in data:
    same_prev_pc = (data['sdo1_prev_distance_postcode'].astype(str).str.strip() == 
                    data['sdo1_previous_Previous_School_Postal_Code'].astype(str).str.strip())
    print(f"Prev-school postcode equality: {same_prev_pc.mean():.1%}")

# Check orientation presence redundancy
if 'has_orientation' in data and 'has_any_orientation' in data:
    orient_agree = (data['has_orientation'].astype('Int64') == 
                    data['has_any_orientation'].astype('Int64')).mean()
    print(f"has_orientation vs has_any_orientation agreement: {orient_agree:.1%}")

# Check SKC presence redundancy
if 'has_skc' in data and 'sdo2_skc_SKC_DATUM_missing_flag' in data:
    inv_missing = 1 - data['sdo2_skc_SKC_DATUM_missing_flag'].astype('Int64')
    skc_agree = (data['has_skc'].astype('Int64') == inv_missing).mean()
    print(f"has_skc vs not-missing SKC date agreement: {skc_agree:.1%}")

### 5.2 Remove Redundant Columns

In [ ]:
# Drop redundant SKC missing-date flag
if 'sdo2_skc_SKC_DATUM_missing_flag' in data.columns:
    data = data.drop(columns=['sdo2_skc_SKC_DATUM_missing_flag'])
    print("✓ Dropped: sdo2_skc_SKC_DATUM_missing_flag")

In [ ]:
# Drop columns no longer needed for modeling
cols_to_drop = [
    "sdo5_degree_degree_start_date",           # Used only for derivation
    "sdo5_degree_enrollment_date",             # Used only for derivation
    "sdo1_previous_Final_Exam_Date",           # Used only for derivation
    "sdo1_characteristics_age_start_study",    # Replaced by age_category
    "sdo1_prev_distance_distance_to_3812_PA",  # Amersfoort campus (not relevant)
    "sdo1_postal_distance_distance_to_3812_PA",
    "sdo2_orientation_STUDENTNUMMER",          # SINH_ID is canonical
    "performance_diff_blocks",                  # Keep only categorical version
    "sdo1_characteristics_student_number"      # Used only for derivation
]

existing = [c for c in cols_to_drop if c in data.columns]
print(f"Dropping {len(existing)} columns: {existing}")

before = data.shape
data = data.drop(columns=existing, errors="ignore")
print(f"Shape: {before} → {data.shape}")

### 5.3 Remove Orphaned Missing Flags

In [ ]:
# Identify all _missing_flag columns
missing_flag_cols = [c for c in data.columns if c.endswith('_missing_flag')]
print(f"Total '_missing_flag' columns: {len(missing_flag_cols)}\n")

# Group by prefix for inspection
grouped = defaultdict(list)
for col in missing_flag_cols:
    prefix = col.split('_')[0]
    grouped[prefix].append(col)

for group, cols in sorted(grouped.items()):
    print(f"{group.upper()} ({len(cols)}): {', '.join([c.split('_')[-3] for c in cols[:3]])}{'...' if len(cols) > 3 else ''}")

In [ ]:
# Drop flags for deleted columns
deleted_cols = [
    "sdo5_degree_degree_start_date",
    "sdo1_characteristics_age_start_study",
    "sdo1_prev_distance_distance_to_3812_PA",
    "sdo1_postal_distance_distance_to_3812_PA",
    "sdo2_orientation_STUDENTNUMMER",
]

flags_to_drop = [f"{col}_missing_flag" for col in deleted_cols if f"{col}_missing_flag" in data.columns]
print(f"Dropping {len(flags_to_drop)} orphaned flags")

before = data.shape
data = data.drop(columns=flags_to_drop, errors="ignore")
print(f"Shape: {before} → {data.shape}")

## 6. Handle Missing Values in Derived Features

### 6.1 Missing Value Strategy

**Structural missingness** (meaningful NaN - do NOT impute):
- Orientation dates & derived gaps → student did not attend
- SKC dates & derived gaps → student did not have SKC meeting  
- Degree de-enrollment date → student still enrolled

**Non-structural missingness** (impute):
- `sdo5_degree_POSTAL_COUNTRY_NL` → delete NaN rows (only 2)
- `gap_prev_exam_to_start_weeks` → impute with mean
- `sdo1_previous_Multiple_Previous_Education` → impute with mean

In [ ]:
# Check top 20 columns with most NaN values
nan_summary = data.isna().sum().sort_values(ascending=False).head(20)
print("📊 Top 20 columns with missing values:")
print(nan_summary)

### 6.2 Delete Rows with Missing Country Code

In [ ]:
print("Before:", data['sdo5_degree_POSTAL_COUNTRY_NL'].value_counts(dropna=False))

In [ ]:
before = len(data)
data = data.dropna(subset=['sdo5_degree_POSTAL_COUNTRY_NL'])
print(f"✓ Deleted {before - len(data)} rows with missing country code")
print("After:", data['sdo5_degree_POSTAL_COUNTRY_NL'].value_counts(dropna=False))

### 6.3 Impute Previous Exam Gap

In [ ]:
gap_mean = data['gap_prev_exam_to_start_weeks'].mean()
data['gap_prev_exam_to_start_weeks'] = data['gap_prev_exam_to_start_weeks'].fillna(gap_mean)
print(f"✓ Imputed gap_prev_exam_to_start_weeks with mean: {gap_mean:.2f} weeks")
print(f"Remaining NaN: {data['gap_prev_exam_to_start_weeks'].isna().sum()}")

### 6.4 Impute Multiple Previous Education

In [ ]:
# Compute mode (mode() returns a Series, so take the first value)
multiple_mode = data['sdo1_previous_Multiple_Previous_Education'].mode()[0]

# Impute with mode
data['sdo1_previous_Multiple_Previous_Education'] = (
    data['sdo1_previous_Multiple_Previous_Education'].fillna(multiple_mode)
)

# Confirm
print(f"✓ Imputed sdo1_previous_Multiple_Previous_Education with mode: {multiple_mode}")
print(f"Remaining NaN: {data['sdo1_previous_Multiple_Previous_Education'].isna().sum()}")

In [ ]:
# Verify remaining NaN values
nan_summary = data.isna().sum().sort_values(ascending=False).head(20)
print("📊 Top 20 columns with remaining NaN values:")
print(nan_summary)

## 7. Create Missing Flags for Structural Missingness

### 7.1 Identify Columns Needing Missing Flags

In [ ]:
# Exclude columns that were already imputed
exclude_from_flags = ['sdo5_degree_POSTAL_COUNTRY_NL', 'gap_prev_exam_to_start_weeks', 
                      'sdo1_previous_Multiple_Previous_Education']

# Find columns with NaN that don't have flags yet
cols_needing_flag = []
for col, n_missing in data.isna().sum().items():
    if n_missing > 0 and col not in exclude_from_flags:
        flag_name = f"{col}_missing_flag"
        if flag_name not in data.columns:
            cols_needing_flag.append((col, n_missing))

print(f"📋 {len(cols_needing_flag)} columns need missing flags:\n")
for col, n_missing in cols_needing_flag[:10]:  # Show first 10
    print(f"  {col}: {n_missing:,} NaNs")
if len(cols_needing_flag) > 10:
    print(f"  ... and {len(cols_needing_flag) - 10} more")

### 7.2 Create Missing Flags

In [ ]:
# Create missing flags for structural missingness
for col, _ in cols_needing_flag:
    flag_name = f"{col}_missing_flag"
    data[flag_name] = data[col].isna().astype(int)

print(f"✓ Created {len(cols_needing_flag)} missing flags")

## 8. Save Final Dataset

In [ ]:
# Save final dataset
output_path = OUT_DIR / "features_derived.csv"
data.to_csv(output_path, index=False)

print(f"✅ Successfully saved features_derived.csv")
print(f"   Path: {output_path}")
print(f"   Shape: {data.shape}")
print(f"   Columns: {len(data.columns)}")